In [ ]:
%%capture
!pip install corus natasha pandas scikit-learn nltk

In [ ]:
!wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

--2026-04-12 17:46:04--  https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/87156914/0b363e00-0126-11e9-9e3c-e8c235463bd6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-12T18%3A40%3A06Z&rscd=attachment%3B+filename%3Dlenta-ru-news.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-12T17%3A39%3A30Z&ske=2026-04-12T18%3A40%3A06Z&sks=b&skv=2018-11-09&sig=vQ1q2uKTxpO2m7kwG4FNs9LWgzXru6BTAx4Q%2B9D27Ms%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NjAxOTQ5NywibmJmIjoxNzc2MDE1ODk3LCJwYXRoIjoicmVsZWFzZWFzc2V0

In [ ]:
from corus import load_lenta

path = 'lenta-ru-news.csv.gz'
records = load_lenta(path)
next(records)

LentaRecord(
    url='https://lenta.ru/news/2018/12/14/cancer/',
    title='Названы регионы России с\xa0самой высокой смертностью от\xa0рака',
    text='Вице-премьер по социальным вопросам Татьяна Голикова рассказала, в каких регионах России зафиксирована наиболее высокая смертность от рака, сообщает РИА Новости. По словам Голиковой, чаще всего онкологические заболевания становились причиной смерти в Псковской, Тверской, Тульской и Орловской областях, а также в Севастополе. Вице-премьер напомнила, что главные факторы смертности в России — рак и болезни системы кровообращения. В начале года стало известно, что смертность от онкологических заболеваний среди россиян снизилась впервые за три года. По данным Росстата, в 2017 году от рака умерли 289 тысяч человек. Это на 3,5 процента меньше, чем годом ранее.',
    topic='Россия',
    tags='Общество',
    date=None
)

In [ ]:
dataset = [next(records).text for i in range(100000)]
dataset[0]

'Австрийские правоохранительные органы не представили доказательств нарушения российскими биатлонистами антидопинговых правил. Об этом сообщил посол России в Вене Дмитрий Любинский по итогам встречи уполномоченного адвоката дипмиссии с представителями прокуратуры страны, передает ТАСС. «Действует презумпция невиновности. Каких-либо ограничений свободы передвижения для команды нет», — добавили в посольстве. Международный союз биатлонистов (IBU) также не будет применять санкции к российским биатлонистам. Все они продолжат выступление на Кубке мира. Полиция нагрянула в отель сборной России в Хохфильцене вечером 12 декабря. Как написал биатлонист Александр Логинов, их считают виновными в махинациях с переливанием крови. Биатлонисту Антону Шипулину, также попавшему в список, полиция нанесла отдельный визит: сейчас он тренируется отдельно в австрийском Обертиллахе. Обвинения спортсмен назвал бредом, а также указал на «охоту на ведьм» в мировом биатлоне. В Австрии прием допинга — уголовное пр

In [ ]:
import re
import pandas as pd

from natasha import Segmenter, NewsEmbedding, NewsMorphTagger, MorphVocab, Doc

In [ ]:
data = []

for i in range(100000):
    record = next(records)
    data.append([record.title, record.text, record.topic])

df = pd.DataFrame(data, columns=["title", "text", "topic"])
df.head()

,title,text,topic
0,«Динамо» в меньшинстве уступило «Наполи» в Лиг...,Московский футбольный клуб «Динамо» уступил ит...,Спорт
1,Минкомсвязи поддержало перевод советских фильм...,Министерство связи и массовых коммуникаций Рос...,Культура
2,МВФ спрогнозировал рост госдолга Украины до 10...,Государственный долг Украины к 2020 году дости...,Экономика
3,Убийцу Усамы бен Ладена взяли на работу в Fox ...,Бывший боец спецподразделения ВМС США «Морские...,Мир
4,В СКР допустили возобновление расследования в ...,Бывший министр обороны Анатолий Сердюков вновь...,Силовые структуры


In [ ]:
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df["topic"] = df["topic"].fillna("unknown")

In [ ]:
df["full_text"] = df["title"] + " " + df["text"]
df.head()

,title,text,topic,full_text
0,«Динамо» в меньшинстве уступило «Наполи» в Лиг...,Московский футбольный клуб «Динамо» уступил ит...,Спорт,«Динамо» в меньшинстве уступило «Наполи» в Лиг...
1,Минкомсвязи поддержало перевод советских фильм...,Министерство связи и массовых коммуникаций Рос...,Культура,Минкомсвязи поддержало перевод советских фильм...
2,МВФ спрогнозировал рост госдолга Украины до 10...,Государственный долг Украины к 2020 году дости...,Экономика,МВФ спрогнозировал рост госдолга Украины до 10...
3,Убийцу Усамы бен Ладена взяли на работу в Fox ...,Бывший боец спецподразделения ВМС США «Морские...,Мир,Убийцу Усамы бен Ладена взяли на работу в Fox ...
4,В СКР допустили возобновление расследования в ...,Бывший министр обороны Анатолий Сердюков вновь...,Силовые структуры,В СКР допустили возобновление расследования в ...


In [ ]:
segmenter = Segmenter()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
morph_vocab = MorphVocab()

In [ ]:
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^а-я\s]", " ", text)

    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)

    words = []

    for token in doc.tokens:
        token.lemmatize(morph_vocab)
        if token.lemma:
            words.append(token.lemma)

    return " ".join(words)

удаляем пробелы, приводим к нижнему регистру, оставляем русские буквы, что бы обработать Наташей, которая является лучшей библиотекой специализированной имеено для русского. Однако выполнлось более часа и если использовать pymorphy будет более бытро.

In [ ]:
df["clean_text"] = df["full_text"].apply(normalize)
df.head()

,title,text,topic,full_text,clean_text
0,«Динамо» в меньшинстве уступило «Наполи» в Лиг...,Московский футбольный клуб «Динамо» уступил ит...,Спорт,«Динамо» в меньшинстве уступило «Наполи» в Лиг...,динамо в меньшинство уступить наполоть в лига ...
1,Минкомсвязи поддержало перевод советских фильм...,Министерство связи и массовых коммуникаций Рос...,Культура,Минкомсвязи поддержало перевод советских фильм...,минкомсвязи поддержать перевод советский фильм...
2,МВФ спрогнозировал рост госдолга Украины до 10...,Государственный долг Украины к 2020 году дости...,Экономика,МВФ спрогнозировал рост госдолга Украины до 10...,мвф спрогнозировать рост госдолг украина до ми...
3,Убийцу Усамы бен Ладена взяли на работу в Fox ...,Бывший боец спецподразделения ВМС США «Морские...,Мир,Убийцу Усамы бен Ладена взяли на работу в Fox ...,убийца усама бен ладен взять на работа в бывши...
4,В СКР допустили возобновление расследования в ...,Бывший министр обороны Анатолий Сердюков вновь...,Силовые структуры,В СКР допустили возобновление расследования в ...,в скр допустить возобновление расследование в ...


In [ ]:
X = df["clean_text"].values
y = df["topic"].values

In [ ]:
counts = df["topic"].value_counts()
df = df[df["topic"].isin(counts[counts >= 2].index)]

In [ ]:
Не дает обучиться на тех статьях, где топик из теста нет в трейне. Поэтому удаляем топики, встречающиеся 1 раз

In [ ]:
from sklearn.model_selection import train_test_split

X_train_0, X_test, y_train_0, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_0, y_train_0, test_size=0.25, stratify=y_train_0, random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
count_vectorizer = CountVectorizer()

X_train_cv = count_vectorizer.fit_transform(X_train)
X_valid_cv = count_vectorizer.transform(X_valid)
X_test_cv = count_vectorizer.transform(X_test)

In [ ]:
cv = LogisticRegression(max_iter=600)

In [ ]:
cv.fit(X_train_count, y_train)

LogisticRegression(max_iter=600)

In [ ]:
pred_cv= cv.predict(X_valid_count)

In [ ]:
print("Accuracy:", accuracy_score(y_valid, pred_cv))

Accuracy: 0.8786


In [ ]:
print("Macro F1:", f1_score(y_valid, pred_cv, average="macro"))

Macro F1: 0.714402022861882


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_valid_tfidf = tfidf_vectorizer.transform(X_valid)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [ ]:
tfidf_model = LogisticRegression(max_iter=600)

In [ ]:
tfidf_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=600)

In [ ]:
tfidf_valid_pred = tfidf_model.predict(X_valid_tfidf)

In [ ]:
print("Accuracy:", accuracy_score(y_valid, tfidf_valid_pred))

Accuracy: 0.87925


In [ ]:
print("Macro F1:", f1_score(y_valid, tfidf_valid_pred, average="macro"))

Macro F1: 0.6412211088663289


In [ ]:
best_count_f1 = 0
best_count_model = None
best_count_vectorizer = None
for min_df in [3, 5]:
    for ngram_range in [(1, 1), (1, 2)]:
        for C in [0.5, 1, 2]:
            vectorizer = CountVectorizer(min_df=min_df, ngram_range=ngram_range)
            X_train_vec = vectorizer.fit_transform(X_train)
            X_valid_vec = vectorizer.transform(X_valid)

            model = LogisticRegression(max_iter=1000, C=C)
            model.fit(X_train_vec, y_train)

            pred = model.predict(X_valid_vec)
            score = f1_score(y_valid, pred, average="macro")

            if score > best_count_f1:
                best_count_f1 = score
                best_count_model = model
                best_count_vectorizer = vectorizer
print("Лучший результат Count:", best_count_f1)

Лучший результат Count: 0.7301232439178251


In [1]:
best_count_valid_pred = best_count_model.predict(best_count_vectorizer.transform(X_valid))

print("Count after tuning")
print("Accuracy:", accuracy_score(y_valid, best_count_valid_pred))
print("Macro F1:", f1_score(y_valid, best_count_valid_pred, average="macro"))

NameError: name 'best_count_model' is not defined

In [ ]:
best_tfidf_f1 = 0
best_tfidf_model = None
best_tfidf_vectorizer = None

In [ ]:
for min_df in [3, 5]:
    for ngram_range in [(1, 1), (1, 2)]:
        for sublinear_tf in [True, False]:
            for C in [0.5, 1, 2]:

                vectorizer = TfidfVectorizer(
                    min_df=min_df,
                    ngram_range=ngram_range,
                    sublinear_tf=sublinear_tf
                )

                X_train_vec = vectorizer.fit_transform(X_train)
                X_valid_vec = vectorizer.transform(X_valid)

                model = LogisticRegression(max_iter=1000, C=C)
                model.fit(X_train_vec, y_train)

                pred = model.predict(X_valid_vec)
                score = f1_score(y_valid, pred, average="macro")

                if score > best_tfidf_f1:
                    best_tfidf_f1 = score
                    best_tfidf_model = model
                    best_tfidf_vectorizer = vectorizer

In [ ]:
print("Лучший результат TF-IDF:", best_tfidf_f1)

In [ ]:
best_tfidf_valid_pred = best_tfidf_model.predict(best_tfidf_vectorizer.transform(X_valid))

print("TF-IDF after tuning")
print("Accuracy:", accuracy_score(y_valid, best_tfidf_valid_pred))
print("Macro F1:", f1_score(y_valid, best_tfidf_valid_pred, average="macro"))

In [ ]:
if best_tfidf_f1 > best_count_f1:
    best_model = best_tfidf_model
    best_vectorizer = best_tfidf_vectorizer
    best_name = "TfidfVectorizer + LogisticRegression"
else:
    best_model = best_count_model
    best_vectorizer = best_count_vectorizer
    best_name = "CountVectorizer + LogisticRegression"

In [ ]:
print("Лучшая модель:", best_name)

In [ ]:
!pip install gensim navec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 21.4 MB/s eta 0:00:00


In [ ]:
train_tokens = []
for text in X_train:
    train_tokens.append(text.split())

valid_tokens = []
for text in X_valid:
    valid_tokens.append(text.split())

test_tokens = []
for text in X_test:
    test_tokens.append(text.split())

In [ ]:
from gensim.models import Word2Vec


In [ ]:
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100, # default = 100
    window=7, # default = 5 предложения в новостях бывают длинные, берем чуть больше стандарта для широкого окна
    min_count=5,
    sg=1, # Training algorithm: 1 for skip-gram
    epochs=5, # достаточно для обучения
    seed=2023, # для стабильности обучения
)

In [ ]:
print(w2v_model.wv.most_similar("россия", topn=10))

[('рф', 0.7341787219047546), ('российский', 0.711721658706665), ('белоруссия', 0.6328052282333374), ('росэнергобанк', 0.6319611072540283), ('снг', 0.6187311410903931), ('страна', 0.6183104515075684), ('экхард', 0.617209792137146), ('ушацкас', 0.6159598231315613), ('вигаудас', 0.6156994104385376), ('наднациональный', 0.6083381772041321)]


In [ ]:
print(w2v_model.wv.most_similar("плен", topn=10))

[('пленник', 0.7417477369308472), ('вызволение', 0.6992713809013367), ('пленить', 0.6861338019371033), ('заложник', 0.6590317487716675), ('пленный', 0.6560925841331482), ('аллекс', 0.6524372100830078), ('убив', 0.6516873240470886), ('кэндзи', 0.64960777759552), ('юкава', 0.6493660807609558), ('гото', 0.6444940567016602)]


In [ ]:
print(w2v_model.wv.doesnt_match(["президент", "петербург", "россия", "петух"]))

петух


Word2Vec даёт векторы только для отдельных слов, а модели типа LogisticRegression нужен один числовой вектор на весь текст. Поэтому берем среднее по векторам слов

In [ ]:
import numpy as np
def get_text_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
X_train_w2v = []
for tokens in train_tokens:
    X_train_w2v.append(get_text_vector(tokens, w2v_model))

X_valid_w2v = []
for tokens in valid_tokens:
    X_valid_w2v.append(get_text_vector(tokens, w2v_model))

X_test_w2v = []
for tokens in test_tokens:
    X_test_w2v.append(get_text_vector(tokens, w2v_model))

In [ ]:
X_train_w2v = np.array(X_train_w2v)
X_valid_w2v = np.array(X_valid_w2v)
X_test_w2v = np.array(X_test_w2v)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
lr_w2v = LogisticRegression(max_iter=600)
lr_w2v.fit(X_train_w2v, y_train)

LogisticRegression(max_iter=600)

обучили LogisticRegression на своих эмбеддингах

In [ ]:
pred_valid_w2v = lr_w2v.predict(X_valid_w2v)

print("W2V valid accuracy:", accuracy_score(y_valid, pred_valid_w2v))
print("W2V valid macro f1:", f1_score(y_valid, pred_valid_w2v, average="macro"))


W2V valid accuracy: 0.8497
W2V valid macro f1: 0.6130212471523502


In [ ]:
from navec import Navec

In [ ]:
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar

--2026-04-12 22:35:35--  https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26634240 (25M) [application/x-tar]
Saving to: ‘navec_news_v1_1B_250K_300d_100q.tar’

navec_news_v1_1B_25 100%[===================>]  25.40M  10.6MB/s    in 2.4s    

2026-04-12 22:35:38 (10.6 MB/s) - ‘navec_news_v1_1B_250K_300d_100q.tar’ saved [26634240/26634240]



In [ ]:
path = 'navec_news_v1_1B_250K_300d_100q.tar'
navec = Navec.load(path)
navec['человек'][:15]

array([-0.13068067, -0.12051002, -0.05782367,  0.07967507,  0.08338855,
        0.59920526,  0.4020081 , -1.0838276 ,  0.12556174,  0.17060532,
        0.16637331, -0.00257014,  0.51296437,  0.17175263, -0.40394753],
      dtype=float32)

In [ ]:
def get_text_vector_navec(tokens, navec_model):
    vectors = []

    for word in tokens:
        if word in navec_model:
            vectors.append(navec_model[word])

    if len(vectors) == 0:
        return np.zeros(300)

    return np.mean(vectors, axis=0)

In [ ]:
X_train_navec = []
for tokens in train_tokens:
    X_train_navec.append(get_text_vector_navec(tokens, navec))

X_valid_navec = []
for tokens in valid_tokens:
    X_valid_navec.append(get_text_vector_navec(tokens, navec))

X_test_navec = []
for tokens in test_tokens:
    X_test_navec.append(get_text_vector_navec(tokens, navec))

In [ ]:
X_train_navec = np.array(X_train_navec)
X_valid_navec = np.array(X_valid_navec)
X_test_navec = np.array(X_test_navec)

In [ ]:
lr_navec = LogisticRegression(max_iter=600)
lr_navec.fit(X_train_navec, y_train)

LogisticRegression(max_iter=600)

обучили LogisticRegression на navec эмбеддингах

In [ ]:
pred_valid_navec = lr_navec.predict(X_valid_navec)

print("Navec valid accuracy:", accuracy_score(y_valid, pred_valid_navec))
print("Navec valid macro f1:", f1_score(y_valid, pred_valid_navec, average="macro"))

Navec valid accuracy: 0.84655
Navec valid macro f1: 0.640137388097725


In [ ]:
w2v_f1 = f1_score(y_valid, pred_valid_w2v, average="macro")
navec_f1 = f1_score(y_valid, pred_valid_navec, average="macro")

print("W2V macro F1:", w2v_f1)
print("Navec macro F1:", navec_f1)

W2V macro F1: 0.6130212471523502
Navec macro F1: 0.640137388097725


navec лучше

In [ ]:
if navec_f1 > w2v_f1:
    pred_test = lr_navec.predict(X_test_navec)
    print("Test accuracy:", accuracy_score(y_test, pred_test))
    print("Test macro f1:", f1_score(y_test, pred_test, average="macro"))
else:
    pred_test = lr_w2v.predict(X_test_w2v)
    print("Test accuracy:", accuracy_score(y_test, pred_test))
    print("Test macro f1:", f1_score(y_test, pred_test, average="macro"))

Test accuracy: 0.8443
Test macro f1: 0.5941351274114715
